# Semantic ICL — Embedding Builder

Build per-stance embedding indexes from the SemEval 2016 Task 6 training set.
These will be used to retrieve semantically similar in-context examples at inference time.

**Output (saved to `data_in/semantic_icl/`):**
- `embeddings_against.npy`, `index_against.json`
- `embeddings_neutral.npy`, `index_neutral.json`
- `embeddings_pro.npy`, `index_pro.json`

## Step 1 — Imports and Config

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Paths
PROJECT_ROOT = Path("..").resolve()
TRAINING_DATA_PATH = PROJECT_ROOT / "data_in" / "semeval2016_task6_stance.csv"
OUTPUT_DIR = PROJECT_ROOT / "data_in" / "semantic_icl"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Embedding model — same one already used in the project
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# Stance label mapping (normalize to lowercase for consistency)
STANCE_CLASSES = ["against", "neutral", "pro"]

print(f"Project root : {PROJECT_ROOT}")
print(f"Training data: {TRAINING_DATA_PATH}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Embedding model: {EMBEDDING_MODEL}")

/Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root : /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper
Training data: /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper/data_in/semeval2016_task6_stance.csv
Output dir   : /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper/data_in/semantic_icl
Embedding model: all-MiniLM-L6-v2


## Step 2 — Load and Inspect Training Data

In [2]:
df = pd.read_csv(TRAINING_DATA_PATH)

# Normalize stance labels to lowercase
df["stance_label"] = df["stance_label"].str.lower().str.strip()

print(f"Total training examples: {len(df)}")
print(f"\nStance distribution:")
print(df["stance_label"].value_counts())
print(f"\nTopics covered:")
print(df["query"].unique())
df.head(3)

Total training examples: 2814

Stance distribution:
stance_label
against    1342
neutral     741
pro         731
Name: count, dtype: int64

Topics covered:
['Atheism' 'Climate Change is a Real Concern' 'Feminist Movement'
 'Hillary Clinton' 'Legalization of Abortion']


,id,content,query,stance_label,stance_label_original,source_dataset
0,101,dear lord thank u for all of ur blessings forg...,Atheism,against,AGAINST,SemEval-2016 Task 6 trainingdata
1,102,"Blessed are the peacemakers, for they shall be...",Atheism,against,AGAINST,SemEval-2016 Task 6 trainingdata
2,103,I am not conformed to this world. I am transfo...,Atheism,against,AGAINST,SemEval-2016 Task 6 trainingdata


## Step 3 — Split by Stance Class

In [3]:
stance_splits = {}

for stance in STANCE_CLASSES:
    subset = df[df["stance_label"] == stance].reset_index(drop=True)
    stance_splits[stance] = subset
    print(f"  {stance:>8}: {len(subset)} examples")

print(f"\nTotal accounted for: {sum(len(v) for v in stance_splits.values())} / {len(df)}")

   against: 1342 examples
   neutral: 741 examples
       pro: 731 examples

Total accounted for: 2814 / 2814


## Step 4 — Load Embedding Model

In [4]:
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded: {EMBEDDING_MODEL}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loaded: all-MiniLM-L6-v2
Embedding dimension: 384


## Step 5 — Embed Each Stance Class and Save

For each stance class we:
1. Embed all documents using sentence-transformers (L2-normalized for cosine similarity)
2. Save embeddings as `.npy`
3. Save a parallel index as `.json` — each entry stores the original row data needed to reconstruct the ICL example at inference time

In [5]:
for stance in STANCE_CLASSES:
    subset = stance_splits[stance]
    texts = subset["content"].tolist()

    print(f"\n[{stance}] Embedding {len(texts)} documents...")
    embeddings = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

    # Save embeddings
    emb_path = OUTPUT_DIR / f"embeddings_{stance}.npy"
    np.save(emb_path, embeddings)

    # Save index — parallel list of records matching embeddings row-for-row
    index = subset[["id", "content", "query", "stance_label"]].to_dict(orient="records")
    idx_path = OUTPUT_DIR / f"index_{stance}.json"
    with open(idx_path, "w") as f:
        json.dump(index, f, indent=2)

    print(f"  Saved embeddings : {emb_path.name}  shape={embeddings.shape}")
    print(f"  Saved index      : {idx_path.name}   records={len(index)}")


[against] Embedding 1342 documents...


Batches: 100%|██████████| 21/21 [00:02<00:00,  9.70it/s]


  Saved embeddings : embeddings_against.npy  shape=(1342, 384)
  Saved index      : index_against.json   records=1342

[neutral] Embedding 741 documents...


Batches: 100%|██████████| 12/12 [00:00<00:00, 20.54it/s]


  Saved embeddings : embeddings_neutral.npy  shape=(741, 384)
  Saved index      : index_neutral.json   records=741

[pro] Embedding 731 documents...


Batches: 100%|██████████| 12/12 [00:00<00:00, 25.80it/s]

  Saved embeddings : embeddings_pro.npy  shape=(731, 384)
  Saved index      : index_pro.json   records=731


## Step 6 — Verify Output Files

In [6]:
print("Files written to:", OUTPUT_DIR)
print()
for f in sorted(OUTPUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<35}  {size_kb:>8.1f} KB")

Files written to: /Users/nityaakalra/Desktop/nyt_topic_modeling/topic_modeling_paper/data_in/semantic_icl

  embeddings_against.npy                 2013.1 KB
  embeddings_neutral.npy                 1111.6 KB
  embeddings_pro.npy                     1096.6 KB
  index_against.json                      286.1 KB
  index_neutral.json                      154.6 KB
  index_pro.json                          154.4 KB


## Step 7 — Sanity Check: Retrieve Nearest Neighbor for a Sample Query

Load the saved files back and run a quick cosine similarity lookup to confirm the retrieval works correctly before we wire this into the main pipeline.

In [9]:
def retrieve_top_k(query_text: str, stance: str, k: int = 3) -> list[dict]:
    """Return top-k most similar training examples for a given stance class."""
    emb_path = OUTPUT_DIR / f"embeddings_{stance}.npy"
    idx_path = OUTPUT_DIR / f"index_{stance}.json"

    embeddings = np.load(emb_path)
    with open(idx_path) as f:
        index = json.load(f)

    query_vec = model.encode([query_text], normalize_embeddings=True)
    scores = (embeddings @ query_vec.T).squeeze()
    top_k_idx = np.argsort(scores)[::-1][:k]

    results = []
    for i in top_k_idx:
        record = index[i].copy()
        record["similarity_score"] = float(scores[i])
        results.append(record)
    return results


# Sample query from the test set
sample_query = "we should not believe in relgion"

print(f"Query: {sample_query!r}\n")
for stance in STANCE_CLASSES:
    top1 = retrieve_top_k(sample_query, stance, k=1)[0]
    print(f"[{stance}]  score={top1['similarity_score']:.4f}")
    print(f"  topic : {top1['query']}")
    print(f"  text  : {top1['content'][:440]}...")
    print()

Query: 'we should not believe in relgion'

[against]  score=0.4830
  topic : Atheism
  text  : I believe that too often religious people do good for the same reasons non-religious do, but they give undo credit to their faith. #SemST...

[neutral]  score=0.4193
  topic : Climate Change is a Real Concern
  text  : @shalyn67 religion / race have already been disproven by reason and science. I wonder what we can do about politics #drought #SemST...

[pro]  score=0.4785
  topic : Atheism
  text  : Religions stopped being credible the minute the first dinosaur fossil was found.   #SemST...



---

## Step 8 — Load Test Set

In [ ]:
TEST_DATA_DIR = PROJECT_ROOT / "data_in" / "semeval2016_task6_testdata_gold"

test_docs = []
for path in sorted(TEST_DATA_DIR.glob("*.json")):
    with open(path) as f:
        test_docs.append(json.load(f))

test_df = pd.DataFrame(test_docs)
test_df["stance_label"] = test_df["stance_label"].str.lower().str.strip()

print(f"Test docs loaded: {len(test_df)}")
print(f"\nStance distribution:")
print(test_df["stance_label"].value_counts())
test_df.head(3)

## Step 9 — Embed All Test Documents (in memory)

In [ ]:
test_texts = test_df["content"].tolist()

print(f"Embedding {len(test_texts)} test documents...")
test_embeddings = model.encode(test_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Done. Shape: {test_embeddings.shape}")

## Step 10 — Load Training Indexes into Memory

Load all 3 stance indexes once so we don't re-read files for every test doc.

In [ ]:
train_embeddings = {}
train_index = {}

for stance in STANCE_CLASSES:
    train_embeddings[stance] = np.load(OUTPUT_DIR / f"embeddings_{stance}.npy")
    with open(OUTPUT_DIR / f"index_{stance}.json") as f:
        train_index[stance] = json.load(f)
    print(f"  {stance:>8}: {train_embeddings[stance].shape[0]} examples loaded")

## Step 11 — Retrieve Top-5 Per Stance for Every Test Document

We retrieve top-5 per stance class (15 total per doc) so we can slice down to any shot count later.
Result: a dict keyed by test doc `_id`, value is a list of 15 retrieved examples (5 per stance, in order).

In [ ]:
TOP_K = 5  # retrieve top-5 per stance to support up to 15-shot

retrieved_all = {}

for i, row in test_df.iterrows():
    doc_id = row["_id"]
    query_vec = test_embeddings[i]  # already normalized

    examples = []
    for stance in STANCE_CLASSES:
        scores = train_embeddings[stance] @ query_vec
        top_k_idx = np.argsort(scores)[::-1][:TOP_K]
        for idx in top_k_idx:
            record = train_index[stance][idx].copy()
            record["similarity_score"] = float(scores[idx])
            examples.append(record)

    retrieved_all[doc_id] = examples  # 15 examples: 5 against, 5 neutral, 5 pro

print(f"Retrieved {TOP_K} examples per stance for {len(retrieved_all)} test documents.")
print(f"Total examples per doc: {TOP_K * len(STANCE_CLASSES)}")

## Step 12 — Save Retrieved Examples for Each Shot Count

Each file slices the top-k per stance from the full retrieved set:
- `retrieved_examples_3.json`  → 1 per stance (3 total)
- `retrieved_examples_6.json`  → 2 per stance (6 total)
- `retrieved_examples_9.json`  → 3 per stance (9 total)
- `retrieved_examples_12.json` → 4 per stance (12 total)
- `retrieved_examples_15.json` → 5 per stance (15 total)

In [ ]:
# 3-shot: top-1 per stance
retrieved_3 = {doc_id: [examples[0], examples[5], examples[10]] for doc_id, examples in retrieved_all.items()}
path_3 = OUTPUT_DIR / "retrieved_examples_3.json"
with open(path_3, "w") as f:
    json.dump(retrieved_3, f, indent=2)
print(f"Saved: {path_3.name}  ({path_3.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 6-shot: top-2 per stance
retrieved_6 = {doc_id: examples[0:2] + examples[5:7] + examples[10:12] for doc_id, examples in retrieved_all.items()}
path_6 = OUTPUT_DIR / "retrieved_examples_6.json"
with open(path_6, "w") as f:
    json.dump(retrieved_6, f, indent=2)
print(f"Saved: {path_6.name}  ({path_6.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 9-shot: top-3 per stance
retrieved_9 = {doc_id: examples[0:3] + examples[5:8] + examples[10:13] for doc_id, examples in retrieved_all.items()}
path_9 = OUTPUT_DIR / "retrieved_examples_9.json"
with open(path_9, "w") as f:
    json.dump(retrieved_9, f, indent=2)
print(f"Saved: {path_9.name}  ({path_9.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 12-shot: top-4 per stance
retrieved_12 = {doc_id: examples[0:4] + examples[5:9] + examples[10:14] for doc_id, examples in retrieved_all.items()}
path_12 = OUTPUT_DIR / "retrieved_examples_12.json"
with open(path_12, "w") as f:
    json.dump(retrieved_12, f, indent=2)
print(f"Saved: {path_12.name}  ({path_12.stat().st_size / 1024:.1f} KB)")

In [ ]:
# 15-shot: top-5 per stance (full set)
retrieved_15 = {doc_id: examples[0:5] + examples[5:10] + examples[10:15] for doc_id, examples in retrieved_all.items()}
path_15 = OUTPUT_DIR / "retrieved_examples_15.json"
with open(path_15, "w") as f:
    json.dump(retrieved_15, f, indent=2)
print(f"Saved: {path_15.name}  ({path_15.stat().st_size / 1024:.1f} KB)")

## Step 13 — Spot Check

Print retrieved examples for a few test docs using the 3-shot file to visually confirm retrieval looks sensible.

In [ ]:
sample_ids = list(retrieved_3.keys())[:3]

for doc_id in sample_ids:
    test_row = test_df[test_df["_id"] == doc_id].iloc[0]
    print(f"TEST DOC [{doc_id}]")
    print(f"  text  : {test_row['content'][:100]}")
    print(f"  topic : {test_row['query']}")
    print(f"  label : {test_row['stance_label']}")
    print(f"  Retrieved examples (3-shot):")
    for ex in retrieved_3[doc_id]:
        print(f"    [{ex['stance_label']:>8}]  score={ex['similarity_score']:.4f}  {ex['content'][:80]}...")
    print()